# Evaluating Prompt Effectiveness

## Overview
This tutorial focuses on methods and techniques for evaluating the effectiveness of prompts in AI language models. We'll explore various metrics for measuring prompt performance and discuss both manual and automated evaluation techniques.

## Motivation
As prompt engineering becomes increasingly crucial in AI applications, it's essential to have robust methods for assessing prompt effectiveness. This enables developers and researchers to optimize their prompts, leading to better AI model performance and more reliable outputs.

## Key Components
1. Metrics for measuring prompt performance
2. Manual evaluation techniques
3. Automated evaluation techniques
4. Practical examples using open-source models with HuggingFace Transformers

## Method Details
We'll start by setting up our environment and introducing key metrics for evaluating prompts. We'll then explore manual evaluation techniques, including human assessment and comparative analysis. Next, we'll delve into automated evaluation methods, utilizing techniques like semantic similarity comparisons and token-based metrics. Throughout the tutorial, we'll provide practical examples using a locally-running Qwen3 model via HuggingFace Transformers to demonstrate these concepts in action.

## Conclusion
By the end of this tutorial, you'll have a comprehensive understanding of how to evaluate prompt effectiveness using both manual and automated techniques. You'll be equipped with practical tools and methods to optimize your prompts, leading to more efficient and accurate AI model interactions.

## Setup

First, let's install the required packages and load the local model in Google Colab with 4-bit quantization.

In [ ]:
# --- Google Colab Setup ---
!pip install -q transformers>=4.51.0 accelerate bitsandbytes torch sentence-transformers scikit-learn numpy

import numpy as np
import torch
from google.colab import drive
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

drive.mount('/content/drive')
CACHE_DIR = "/content/drive/MyDrive/models"
MODEL_NAME = "Qwen/Qwen3-8B"

quantization_config = BitsAndBytesConfig(
    load_in_4bit=True, bnb_4bit_compute_dtype=torch.bfloat16, bnb_4bit_quant_type="nf4",
)
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, cache_dir=CACHE_DIR)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME, device_map="auto", quantization_config=quantization_config,
    cache_dir=CACHE_DIR, torch_dtype="auto",
)

# Embedding model for semantic similarity evaluation
embedding_model = SentenceTransformer('all-MiniLM-L6-v2', cache_folder=CACHE_DIR)

def generate(messages, max_new_tokens=512, temperature=0.7, do_sample=True):
    """Generate a response from the model given a list of chat messages."""
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True, enable_thinking=False)
    inputs = tokenizer([text], return_tensors="pt").to(model.device)
    output_ids = model.generate(
        **inputs, max_new_tokens=max_new_tokens,
        temperature=temperature if do_sample else None, do_sample=do_sample, top_k=20,
    )
    generated = output_ids[0][inputs.input_ids.shape[1]:]
    return tokenizer.decode(generated, skip_special_tokens=True)

def count_tokens(text):
    """Count tokens using the model's tokenizer."""
    return len(tokenizer.encode(text))

def semantic_similarity(text1, text2):
    """Calculate semantic similarity between two texts using cosine similarity."""
    embeddings = embedding_model.encode([text1, text2])
    return cosine_similarity([embeddings[0]], [embeddings[1]])[0][0]

## Embedding Spaces Explained

Before we evaluate prompts, we need to understand the core technology that powers semantic evaluation: **sentence embeddings**.

### From Words to Vectors
A sentence embedding compresses an entire sentence into a single dense vector — typically 384 to 768 dimensions — that captures its *meaning*. This is fundamentally different from token (word) embeddings:

| Level | What it captures | Example |
|-------|-----------------|---------|
| **Token embedding** | Meaning of a single word | "bank" → one vector (but which meaning?) |
| **Sentence embedding** | Composite meaning of the whole sentence | "I deposited money at the bank" → one vector |

The key insight: **similar sentences produce similar vectors**. If two sentences mean roughly the same thing, their embedding vectors will point in roughly the same direction in high-dimensional space. This is what makes semantic evaluation possible — we can *measure* meaning similarity mathematically.

### How It Works Under the Hood
The `all-MiniLM-L6-v2` model we loaded above is a 6-layer transformer trained specifically for this task. It:
1. Tokenizes the input sentence into subword tokens
2. Passes tokens through 6 transformer layers (self-attention + feed-forward)
3. Applies **mean pooling** over all token outputs to produce one vector
4. The result is a 384-dimensional vector that encodes the sentence's semantic content

In [ ]:
# Step-by-step: How sentence embeddings work

# Step 1: Take a sentence
sentence = "Machine learning is a subset of artificial intelligence."
print(f"Input sentence: {sentence}")
print(f"Character length: {len(sentence)}")

# Step 2: Encode it into a vector
vector = embedding_model.encode([sentence])[0]
print(f"\nEmbedding shape: {vector.shape}  (384 dimensions)")
print(f"First 10 dimensions: {vector[:10].round(4)}")
print(f"Vector magnitude: {np.linalg.norm(vector):.4f}")

# Step 3: Compare similar vs different sentences
similar_sentence = "AI includes machine learning as one of its branches."
different_sentence = "The weather in Paris is sunny today."

vec_similar = embedding_model.encode([similar_sentence])[0]
vec_different = embedding_model.encode([different_sentence])[0]

# Cosine similarity
sim_similar = cosine_similarity([vector], [vec_similar])[0][0]
sim_different = cosine_similarity([vector], [vec_different])[0][0]

print(f"\n--- Similarity Comparison ---")
print(f'Original:  "{sentence}"')
print(f'Similar:   "{similar_sentence}"  => cosine sim = {sim_similar:.4f}')
print(f'Different: "{different_sentence}"  => cosine sim = {sim_different:.4f}')
print(f"\nThe similar sentence scores {sim_similar - sim_different:.4f} higher — the embedding space clusters meaning.")

### Cosine Similarity: Measuring the Angle Between Meanings

Cosine similarity measures the **angle** between two vectors, ignoring magnitude:

$$\cos(\theta) = \frac{\mathbf{A} \cdot \mathbf{B}}{\|\mathbf{A}\| \|\mathbf{B}\|}$$

| Value | Meaning | Example |
|-------|---------|---------|
| **1.0** | Identical direction = same meaning | "The cat sat on the mat" vs "The cat sat on the mat" |
| **0.7–0.9** | Similar direction = related meaning | "The cat sat on the mat" vs "A feline rested on the rug" |
| **0.0–0.3** | Orthogonal = unrelated topics | "The cat sat on the mat" vs "Stock prices rose sharply" |
| **-1.0** | Opposite direction (rare for embeddings) | Almost never occurs with sentence transformers |

In practice, sentence transformer embeddings are almost always positive (0.0 to 1.0) because the model is trained to place all text in a positive region of the embedding space. A score above **0.7** typically indicates strong semantic similarity.

## Metrics for Measuring Prompt Performance

Let's define some key metrics for evaluating prompt effectiveness, including token-based evaluation:

In [ ]:
def relevance_score(response, expected_content):
    """Calculate relevance score based on semantic similarity to expected content."""
    return semantic_similarity(response, expected_content)

def consistency_score(responses):
    """Calculate consistency score based on similarity between multiple responses."""
    if len(responses) < 2:
        return 1.0  # Perfect consistency if there's only one response
    similarities = []
    for i in range(len(responses)):
        for j in range(i+1, len(responses)):
            similarities.append(semantic_similarity(responses[i], responses[j]))
    return np.mean(similarities)

def specificity_score(response):
    """Calculate specificity score based on response length and unique word count."""
    words = response.split()
    unique_words = set(words)
    return len(unique_words) / len(words) if words else 0

def evaluate_response(response, reference=None):
    """Comprehensive response evaluation with multiple metrics."""
    metrics = {
        "word_count": len(response.split()),
        "token_count": count_tokens(response),
        "unique_word_ratio": len(set(response.lower().split())) / max(len(response.split()), 1),
    }
    if reference:
        # Semantic similarity with reference
        emb_response = embedding_model.encode([response])
        emb_reference = embedding_model.encode([reference])
        metrics["semantic_similarity"] = float(cosine_similarity(emb_response, emb_reference)[0][0])
    return metrics

## Why These Metrics Work (And When They Don't)

Every evaluation metric has blind spots. Understanding these blind spots is essential for building trustworthy evaluation pipelines.

### The Metric Landscape

| Metric | What it measures | Strengths | Blind spots |
|--------|-----------------|-----------|-------------|
| **Cosine similarity** | Semantic overlap between embeddings | Captures paraphrases; language-agnostic meaning | Can be fooled by superficial topic overlap; misses factual errors within the same topic |
| **BLEU / ROUGE** | N-gram (word sequence) overlap | Fast; reproducible; good for translation | Fails completely on valid paraphrases; penalizes creative but correct responses |
| **LLM-as-judge** | Holistic quality assessment | Captures nuance; can evaluate style, tone, correctness | Position bias (prefers first option); length bias (prefers longer); self-preference (favors its own outputs) |
| **Keyword match** | Presence of required terms | Simple; interpretable; deterministic | Ignores context; "not supervised learning" matches "supervised learning" |

The core problem: **no single metric captures "quality."** Each metric is a projection of a high-dimensional concept (response quality) onto a single number. Information is inevitably lost.

In [ ]:
# Demonstrating metric failures: when metrics disagree

# Case 1: Semantically identical, lexically different
#   Should be "same quality" — but word-overlap metrics will say "different"
sent_a = "Supervised learning uses labeled training data to learn a mapping from inputs to outputs."
sent_b = "In the supervised paradigm, algorithms are trained on annotated examples to predict target values from features."

# Case 2: Lexically similar, semantically different
#   Should be "different quality" — but word-overlap metrics will say "similar"
sent_c = "The bank approved the loan for the new house."
sent_d = "The bank of the river was eroded by the new current."

# Cosine similarity (semantic)
sim_case1 = semantic_similarity(sent_a, sent_b)
sim_case2 = semantic_similarity(sent_c, sent_d)

# Simple BLEU-like n-gram overlap (unigram precision)
def unigram_overlap(text1, text2):
    """Simple unigram overlap ratio as a BLEU proxy."""
    words1 = set(text1.lower().split())
    words2 = set(text2.lower().split())
    overlap = words1 & words2
    return len(overlap) / max(len(words1), 1)

bleu_case1 = unigram_overlap(sent_a, sent_b)
bleu_case2 = unigram_overlap(sent_c, sent_d)

print("=== Case 1: Same meaning, different words ===")
print(f'  A: "{sent_a}"')
print(f'  B: "{sent_b}"')
print(f"  Cosine similarity: {sim_case1:.4f}  (HIGH — correctly detects same meaning)")
print(f"  Unigram overlap:   {bleu_case1:.4f}  (LOW — fooled by different vocabulary)")

print(f"\n=== Case 2: Different meaning, similar words ===")
print(f'  C: "{sent_c}"')
print(f'  D: "{sent_d}"')
print(f"  Cosine similarity: {sim_case2:.4f}  (LOWER — detects different meaning)")
print(f"  Unigram overlap:   {bleu_case2:.4f}  (HIGHER — fooled by shared words)")

print(f"\n--- Key insight ---")
print(f"Cosine similarity correctly distinguishes both cases.")
print(f"Unigram overlap gets them backwards — this is why BLEU/ROUGE alone are insufficient.")

### The Lesson: Always Use Multiple Metrics

The demonstration above shows why **no single metric should be trusted in isolation**:

- **Cosine similarity** correctly identified that Case 1 sentences mean the same thing and Case 2 sentences mean different things
- **Unigram overlap** (a BLEU proxy) got both cases wrong — it penalized valid paraphrases and rewarded superficial word sharing

In practice, robust evaluation combines:
1. **Semantic similarity** (cosine) — does it mean the right thing?
2. **Keyword/entity matching** — does it contain required information?
3. **Format compliance** — does it follow structural requirements?
4. **Length compliance** — is it within acceptable bounds?

We'll build this multi-metric framework in the next section.

## Manual Evaluation Techniques

Manual evaluation involves human assessment of prompt-response pairs. Let's create a function to simulate this process:

In [ ]:
def manual_evaluation(prompt, response, criteria):
    """Simulate manual evaluation of a prompt-response pair."""
    print(f"Prompt: {prompt}")
    print(f"Response: {response}")
    print("\nEvaluation Criteria:")
    for criterion in criteria:
        score = float(input(f"Score for {criterion} (0-10): "))
        print(f"{criterion}: {score}/10")
    print("\nAdditional Comments:")
    comments = input("Enter any additional comments: ")
    print(f"Comments: {comments}")

# Example usage
prompt = "Explain the concept of machine learning in simple terms."
messages = [{"role": "user", "content": prompt}]
response = generate(messages)
criteria = ["Clarity", "Accuracy", "Simplicity"]
manual_evaluation(prompt, response, criteria)

## Automated Evaluation Techniques

Now, let's implement some automated evaluation techniques using the local model:

In [ ]:
def automated_evaluation(prompt, response, expected_content):
    """Perform automated evaluation of a prompt-response pair."""
    relevance = relevance_score(response, expected_content)
    specificity = specificity_score(response)
    token_metrics = evaluate_response(response, reference=expected_content)

    print(f"Prompt: {prompt}")
    print(f"Response: {response}")
    print(f"\nRelevance Score: {relevance:.2f}")
    print(f"Specificity Score: {specificity:.2f}")
    print(f"Token Count: {token_metrics['token_count']}")
    print(f"Unique Word Ratio: {token_metrics['unique_word_ratio']:.2f}")

    return {"relevance": relevance, "specificity": specificity, **token_metrics}

# Example usage
prompt = "What are the three main types of machine learning?"
expected_content = "The three main types of machine learning are supervised learning, unsupervised learning, and reinforcement learning."
messages = [{"role": "user", "content": prompt}]
response = generate(messages)
automated_evaluation(prompt, response, expected_content)

## Building a Robust Evaluation Pipeline

Now we combine everything into a structured evaluation framework. A robust pipeline needs:
- **Multiple metrics** that check different quality dimensions
- **Multiple test cases** to avoid overfitting to a single example
- **Statistical aggregation** because single scores are noisy — we need means, standard deviations, and confidence intervals

In [ ]:
# Comprehensive evaluation framework

def keyword_match_score(response, required_keywords):
    """Check what fraction of required keywords appear in the response."""
    response_lower = response.lower()
    matches = sum(1 for kw in required_keywords if kw.lower() in response_lower)
    return matches / len(required_keywords) if required_keywords else 0.0

def length_compliance_score(response, min_words=20, max_words=300):
    """Score how well the response stays within length bounds. 1.0 = within bounds."""
    word_count = len(response.split())
    if min_words <= word_count <= max_words:
        return 1.0
    elif word_count < min_words:
        return word_count / min_words
    else:
        return max_words / word_count

def format_compliance_score(response, required_elements=None):
    """Check for required formatting elements (bullet points, numbered lists, etc.)."""
    if not required_elements:
        return 1.0
    checks = {
        "bullet_points": any(line.strip().startswith(("-", "*")) for line in response.split("\n")),
        "numbered_list": any(line.strip()[:2].rstrip(".").isdigit() for line in response.split("\n") if line.strip()),
        "paragraphs": response.count("\n\n") >= 1,
    }
    matched = sum(1 for elem in required_elements if checks.get(elem, False))
    return matched / len(required_elements)

def evaluate_comprehensive(response, reference, required_keywords=None,
                           min_words=20, max_words=300, required_format=None):
    """Multi-metric evaluation returning a dictionary of scores."""
    scores = {}
    scores["semantic_similarity"] = float(semantic_similarity(response, reference))
    scores["keyword_match"] = keyword_match_score(response, required_keywords or [])
    scores["length_compliance"] = length_compliance_score(response, min_words, max_words)
    scores["format_compliance"] = format_compliance_score(response, required_format)
    scores["specificity"] = specificity_score(response)
    return scores

# Define test cases — each is a complete evaluation scenario
test_cases = [
    {
        "name": "ML types",
        "prompt": "List and briefly explain the three main types of machine learning.",
        "reference": "The three main types are supervised learning (learns from labeled data), unsupervised learning (finds patterns in unlabeled data), and reinforcement learning (learns through reward signals).",
        "keywords": ["supervised", "unsupervised", "reinforcement"],
        "format": ["numbered_list"],
    },
    {
        "name": "Overfitting",
        "prompt": "Explain overfitting in machine learning and how to prevent it.",
        "reference": "Overfitting occurs when a model learns noise in training data, performing well on training data but poorly on unseen data. Prevention methods include regularization, cross-validation, and using more training data.",
        "keywords": ["overfitting", "training", "regularization"],
        "format": ["paragraphs"],
    },
    {
        "name": "Neural networks",
        "prompt": "What is a neural network? Explain using a simple analogy.",
        "reference": "A neural network is a computing system inspired by biological neurons. It consists of layers of interconnected nodes that process information, learning to recognize patterns through training.",
        "keywords": ["neural", "layers", "learn"],
        "format": ["paragraphs"],
    },
    {
        "name": "Bias-variance",
        "prompt": "Explain the bias-variance tradeoff in simple terms.",
        "reference": "The bias-variance tradeoff is the tension between a model being too simple (high bias, underfitting) and too complex (high variance, overfitting). The goal is finding the right balance.",
        "keywords": ["bias", "variance", "tradeoff"],
        "format": ["paragraphs"],
    },
    {
        "name": "Gradient descent",
        "prompt": "How does gradient descent work? Use a mountain analogy.",
        "reference": "Gradient descent is an optimization algorithm that iteratively adjusts parameters by moving in the direction of steepest decrease of the loss function, like walking downhill to find the lowest valley.",
        "keywords": ["gradient", "descent", "loss"],
        "format": ["paragraphs"],
    },
]

def run_evaluation_pipeline(prompt_template, test_cases, label="default"):
    """Run a full evaluation pipeline across all test cases and aggregate results."""
    all_scores = []
    sep = "=" * 60
    print(f"\n{sep}")
    print(f"  Evaluation Pipeline: {label}")
    print(sep)

    for tc in test_cases:
        if "{question}" in prompt_template:
            prompt = prompt_template.format(question=tc["prompt"])
        else:
            prompt = tc["prompt"]
        messages = [{"role": "user", "content": prompt}]
        response = generate(messages, max_new_tokens=256, temperature=0.7)

        scores = evaluate_comprehensive(
            response, tc["reference"],
            required_keywords=tc.get("keywords"),
            required_format=tc.get("format"),
        )
        all_scores.append(scores)
        print(f"\n  [{tc['name']}] sem_sim={scores['semantic_similarity']:.3f}  kw={scores['keyword_match']:.2f}  len={scores['length_compliance']:.2f}  fmt={scores['format_compliance']:.2f}")

    # Statistical aggregation
    metrics = list(all_scores[0].keys())
    dash = "-" * 60
    print(f"\n{dash}")
    print(f"  Aggregated Results ({len(test_cases)} test cases)")
    print(dash)
    summary = {}
    for m in metrics:
        values = [s[m] for s in all_scores]
        mean = np.mean(values)
        std = np.std(values, ddof=1) if len(values) > 1 else 0.0
        ci_95 = 1.96 * std / np.sqrt(len(values))
        summary[m] = {"mean": mean, "std": std, "ci_95": ci_95}
        print(f"  {m:25s}  mean={mean:.4f}  std={std:.4f}  95%CI=+/-{ci_95:.4f}")

    return summary, all_scores

print("Evaluation framework defined. Ready to run pipeline.")

In [ ]:
# Compare two prompt variants using the evaluation pipeline

# Variant A: Simple direct question (zero-shot style)
summary_a, scores_a = run_evaluation_pipeline(
    prompt_template="{question}",
    test_cases=test_cases,
    label="Variant A: Direct question"
)

# Variant B: Instructed prompt with role and constraints
summary_b, scores_b = run_evaluation_pipeline(
    prompt_template="You are an expert ML instructor. Answer concisely in 2-4 sentences. {question}",
    test_cases=test_cases,
    label="Variant B: Instructed with role + constraints"
)

# Head-to-head comparison
sep = "=" * 60
dash = chr(9472) * 60  # unicode box-drawing horizontal line
print(f"\n{sep}")
print("  HEAD-TO-HEAD COMPARISON")
print(sep)
print(f"  {'Metric':<24s} {'Variant A':>12s}   {'Variant B':>12s}   {'Winner':>6s}")
print(f"  {dash[:55]}")
for metric in summary_a:
    a_val = summary_a[metric]["mean"]
    b_val = summary_b[metric]["mean"]
    winner = "A" if a_val > b_val else "B" if b_val > a_val else "tie"
    print(f"  {metric:<24s} {a_val:>10.4f}   {b_val:>10.4f}   {winner:>6s}")

### Inter-Rater Reliability: Can We Trust These Numbers?

A critical question for any evaluation system: **if human judges disagree, how can we trust automated metrics?**

Research consistently shows:
- Human judges agree with each other about **60–80%** of the time on open-ended quality assessments (Cohen's κ ≈ 0.4–0.6)
- Automated metrics correlate with human judgment at varying levels:
  - Cosine similarity: **moderate** correlation (r ≈ 0.5–0.7) — good for topical relevance, weaker for factual accuracy
  - BLEU/ROUGE: **weak-to-moderate** correlation (r ≈ 0.3–0.5) — better for translation than open-ended generation
  - LLM-as-judge: **moderate-to-strong** correlation (r ≈ 0.6–0.8) — best available, but has systematic biases

The key insight: **automated metrics are most useful for relative comparisons** ("Prompt A is better than Prompt B") rather than absolute quality judgments ("This response is good"). Our pipeline uses multiple metrics precisely because each one captures a different facet of quality, and their agreement strengthens our confidence.

## Comparative Analysis

Let's compare the effectiveness of different prompts for the same task using the local model:

In [ ]:
def compare_prompts(prompts, expected_content):
    """Compare the effectiveness of multiple prompts for the same task."""
    results = []
    for prompt in prompts:
        messages = [{"role": "user", "content": prompt}]
        response = generate(messages)
        evaluation = automated_evaluation(prompt, response, expected_content)
        results.append({"prompt": prompt, **evaluation})

    # Sort results by relevance score
    sorted_results = sorted(results, key=lambda x: x['relevance'], reverse=True)

    print("Prompt Comparison Results:")
    for i, result in enumerate(sorted_results, 1):
        print(f"\n{i}. Prompt: {result['prompt']}")
        print(f"   Relevance: {result['relevance']:.2f}")
        print(f"   Specificity: {result['specificity']:.2f}")

    return sorted_results

# Example usage
prompts = [
    "List the types of machine learning.",
    "What are the main categories of machine learning algorithms?",
    "Explain the different approaches to machine learning."
]
expected_content = "The main types of machine learning are supervised learning, unsupervised learning, and reinforcement learning."
compare_prompts(prompts, expected_content)

## Cross-Technique Evaluation

The true power of a structured evaluation pipeline is comparing **different prompting techniques** on the same tasks. This transforms subjective preferences ("I think few-shot works better") into measurable differences ("few-shot scored 0.85 vs 0.72 on semantic similarity across 5 test cases").

In [ ]:
# Cross-technique comparison: zero-shot vs few-shot vs system message vs temperature

# Technique 1: Zero-shot (plain question)
print("Running Technique 1: Zero-shot...")
results_zeroshot = []
for tc in test_cases:
    messages = [{"role": "user", "content": tc["prompt"]}]
    response = generate(messages, max_new_tokens=256, temperature=0.7)
    scores = evaluate_comprehensive(response, tc["reference"],
                                     required_keywords=tc.get("keywords"),
                                     required_format=tc.get("format"))
    results_zeroshot.append(scores)

# Technique 2: Few-shot (provide an example before the question)
print("Running Technique 2: Few-shot...")
results_fewshot = []
few_shot_example = (
    "Q: What is a decision tree?\n"
    "A: A decision tree is a supervised learning algorithm that splits data into "
    "branches based on feature values, creating a tree-like structure of decisions. "
    "Each leaf node represents a prediction.\n\n"
)
for tc in test_cases:
    messages = [{"role": "user", "content": few_shot_example + "Q: " + tc["prompt"] + "\nA:"}]
    response = generate(messages, max_new_tokens=256, temperature=0.7)
    scores = evaluate_comprehensive(response, tc["reference"],
                                     required_keywords=tc.get("keywords"),
                                     required_format=tc.get("format"))
    results_fewshot.append(scores)

# Technique 3: With system message
print("Running Technique 3: System message...")
results_system = []
for tc in test_cases:
    messages = [
        {"role": "system", "content": "You are an expert machine learning instructor. Give clear, concise, and accurate explanations."},
        {"role": "user", "content": tc["prompt"]}
    ]
    response = generate(messages, max_new_tokens=256, temperature=0.7)
    scores = evaluate_comprehensive(response, tc["reference"],
                                     required_keywords=tc.get("keywords"),
                                     required_format=tc.get("format"))
    results_system.append(scores)

# Technique 4: Low temperature (more deterministic)
print("Running Technique 4: Low temperature (0.2)...")
results_lowtemp = []
for tc in test_cases:
    messages = [{"role": "user", "content": tc["prompt"]}]
    response = generate(messages, max_new_tokens=256, temperature=0.2)
    scores = evaluate_comprehensive(response, tc["reference"],
                                     required_keywords=tc.get("keywords"),
                                     required_format=tc.get("format"))
    results_lowtemp.append(scores)

# Build comparison table
techniques = {
    "Zero-shot":      results_zeroshot,
    "Few-shot":       results_fewshot,
    "System msg":     results_system,
    "Low temp (0.2)": results_lowtemp,
}

metrics_list = list(results_zeroshot[0].keys())

sep = "=" * 80
dash = "-" * 80
print(f"\n{sep}")
print("  CROSS-TECHNIQUE COMPARISON TABLE")
print(sep)

# Header
header = f"  {'Technique':<18s}"
for m in metrics_list:
    header += f" {m[:12]:>12s}"
print(header)
print(dash)

# Rows
for tech_name, tech_results in techniques.items():
    row = f"  {tech_name:<18s}"
    for m in metrics_list:
        values = [s[m] for s in tech_results]
        mean_val = np.mean(values)
        row += f" {mean_val:>12.4f}"
    print(row)

print(dash)

# Find best technique per metric
print("\n  Best technique per metric:")
for m in metrics_list:
    best_tech = max(techniques.keys(), key=lambda t: np.mean([s[m] for s in techniques[t]]))
    best_val = np.mean([s[m] for s in techniques[best_tech]])
    print(f"    {m:<25s} -> {best_tech} ({best_val:.4f})")

### What the Numbers Tell Us

This comparison demonstrates the practical value of systematic evaluation:

- **"I think this prompt is better"** becomes **"this technique scored 0.85 vs 0.72 on semantic similarity across 5 test cases"**
- Different techniques may win on different metrics — the "best" technique depends on what you optimize for
- Low temperature typically improves consistency but may reduce creativity
- System messages and few-shot examples often improve keyword coverage because they prime the model for the expected domain

The evaluation pipeline turns intuition into evidence.

## Beyond Automated Metrics

All automated metrics have fundamental limitations. Understanding these limitations is what separates rigorous evaluation from cargo-cult metrics.

### Goodhart's Law
> *"When a measure becomes a target, it ceases to be a good measure."*

If you optimize prompts purely for cosine similarity, you'll eventually find prompts that score high on cosine similarity but produce low-quality responses. The metric becomes the target rather than a proxy for quality.

### The Alignment Problem
What we can **measure** (word overlap, embedding similarity, format compliance) ≠ what we **care about** (helpfulness, truthfulness, safety, clarity). Automated metrics are at best *correlated* with the qualities we care about — they don't directly measure them.

### Human Evaluation: Gold Standard with Caveats
Human evaluation remains the gold standard, but comes with its own problems:
- **Expensive**: Each evaluation requires human time and attention
- **Subjective**: Different evaluators may disagree significantly
- **Inconsistent**: The same evaluator may give different scores on different days
- **Not scalable**: Can't evaluate thousands of prompt variants

In [ ]:
# Human evaluation helper: structured scoring interface
# This generates responses and formats them for manual comparison

def human_eval_compare(prompt, variants, reference=None):
    """Generate responses for multiple prompt variants and format for human scoring."""
    sep = "=" * 70
    print(sep)
    print("  HUMAN EVALUATION SCORECARD")
    print(sep)
    print(f"\n  Task: {prompt}")
    if reference:
        print(f"  Reference: {reference}")
    print()

    responses = []
    for i, (variant_name, messages) in enumerate(variants.items(), 1):
        response = generate(messages, max_new_tokens=256, temperature=0.7)
        responses.append((variant_name, response))

        # Display formatted response for human review
        resp_lines = response.split("\n")[:8]
        print(f"  +-- Variant {i}: {variant_name}")
        print(f"  |")
        for line in resp_lines:
            print(f"  |  {line}")
        if len(response.split("\n")) > 8:
            print(f"  |  ... (truncated)")
        print(f"  |")
        print(f"  |  Score (1-5):  Accuracy [  ]  Clarity [  ]  Completeness [  ]  Conciseness [  ]")
        print(f"  +--")
        print()

    # Also show automated scores for comparison
    if reference:
        print(f"  +-- Automated Scores (for comparison)")
        print(f"  |  {'Variant':<20s} {'Semantic Sim':>12s} {'Specificity':>12s}")
        for variant_name, response in responses:
            sem_sim = semantic_similarity(response, reference)
            spec = specificity_score(response)
            print(f"  |  {variant_name:<20s} {sem_sim:>12.4f} {spec:>12.4f}")
        print(f"  +--")

    print("\n  Instructions: Fill in scores 1-5 for each criterion.")
    print("  Compare automated scores with your human judgment — do they agree?")
    return responses

# Example: Compare two variants for human evaluation
eval_prompt = "Explain what a neural network is in simple terms."
eval_reference = (
    "A neural network is a computing system inspired by biological neurons, "
    "consisting of layers of interconnected nodes that learn to recognize patterns from data."
)

variants = {
    "Direct question": [{"role": "user", "content": eval_prompt}],
    "With system msg": [
        {"role": "system", "content": "Explain concepts simply, as if to a high school student."},
        {"role": "user", "content": eval_prompt}
    ],
}

human_eval_compare(eval_prompt, variants, reference=eval_reference)

### Practical Recommendation: The Two-Stage Approach

Based on these tradeoffs, here is the recommended workflow for production prompt engineering:

**Stage 1 — Rapid Iteration (Automated)**
1. Define your test cases with references and required keywords
2. Use the multi-metric evaluation pipeline to compare prompt variants
3. Filter down to the top 2–3 candidates based on aggregate scores
4. This stage is fast and cheap — run it dozens of times

**Stage 2 — Final Quality Gate (Human)**
1. Take the top candidates from Stage 1
2. Generate responses on a held-out test set (cases not used in Stage 1)
3. Have 2–3 human evaluators score them using the structured scorecard above
4. Select the final prompt based on human consensus

This approach gives you the **speed of automation** for exploration and the **reliability of human judgment** for final decisions. Neither alone is sufficient.

## Putting It All Together

Now, let's create a comprehensive prompt evaluation function that combines both manual and automated techniques:

In [ ]:
def evaluate_prompt(prompt, expected_content, manual_criteria=["Clarity", "Accuracy", "Relevance"]):
    """Perform a comprehensive evaluation of a prompt using both manual and automated techniques."""
    messages = [{"role": "user", "content": prompt}]
    response = generate(messages)

    print("Automated Evaluation:")
    auto_results = automated_evaluation(prompt, response, expected_content)

    print("\nManual Evaluation:")
    manual_evaluation(prompt, response, manual_criteria)

    return {"prompt": prompt, "response": response, **auto_results}

# Example usage
prompt = "Explain the concept of overfitting in machine learning."
expected_content = "Overfitting occurs when a model learns the training data too well, including its noise and fluctuations, leading to poor generalization on new, unseen data."
evaluate_prompt(prompt, expected_content)